# CO2-Aware Workload Migration: Where Should SURF Run Its Workload?

*Use case 3 of 3, about 10 minutes.*

We envision digital twins becoming primary instruments in datacenter operations, continuously and autonomously helping
with major operational decisions and with adapting ICT infrastructure, live, with a human-in-the-loop.
Here, OpenDT is an advisor-on-the-side for workload migration, CO2 aware: the same SURF workload, on the same hosts, on
the grids of the Netherlands (NL), Germany (DE), Romania (RO), and Poland (PL).

In [ ]:
import pandas as pd
from resources import *

check_setup()

## 1. Carbon Intensity of Four Grids

Carbon intensity [gCO2/kWh] from Electricity Maps (October 2022, hourly, ODbL; see `workload/SURF/carbon_zones/LICENSE.md`).

In [ ]:
ask_question("3.1")

In [ ]:
week = {zone: carbon_intensity(zone)[TRACE_START : TRACE_END - pd.Timedelta("1h")] for zone in CARBON_ZONES}
intensity = pd.DataFrame(week)  # 168 hourly values per grid

ax = intensity.plot()
ax.set(xlabel="Time [UTC]", ylabel="Carbon Intensity [gCO2/kWh]", ylim=(0, None))
ax.yaxis.set_major_formatter("{x:,.0f}")  # 1,000
legend_above(ax)
pd.DataFrame({"Average [gCO2/kWh]": intensity.mean()}).round()

In [ ]:
give_solution("3.1", show=False)  # show=True shows the solution

In [ ]:
ask_question("3.2")

In [ ]:
give_solution("3.2", show=False)  # show=True shows the solution

## 2. One Simulation per Grid

Only the carbon intensity trace changes between the simulations (`powerSource.carbonTracePath` in the topology).

In [ ]:
ask_question("3.3")

In [ ]:
runs = simulate_many("carbon", CARBON_ZONES, until=TRACE_END)  # one simulation of the full week per grid

table = pd.DataFrame([run.summary() for run in runs.values()], index=list(runs))
table[["energy_kwh", "co2_kg"]].round()

In [ ]:
give_solution("3.3", show=False)  # show=True shows the solution

In [ ]:
co2 = table["co2_kg"]
ax = co2.sort_values(ascending=False).plot.barh(figsize=(8, 2.5))
ax.set(xlabel="CO2 Emissions [kg]")
ax.xaxis.set_major_formatter("{x:,.0f}")

best = co2.idxmin()
saving = co2["NL"] - co2[best]
print(f"Advice: run the SURF workload in {best}, with {saving:,.0f} kg CO2 less per week than in NL "
      f"({saving / co2['NL']:.0%}).")

## 3. The Human-in-the-Loop

![Datacenter digital twinning](resources/images/digital_twinning.png)

*High-level overview of datacenter digital twinning; a human-in-the-loop (HITL) oversees the process (OpenDT paper,
Figure 1).*

In [ ]:
ask_question("3.4")

In [ ]:
give_solution("3.4", show=False)  # show=True shows the solution

In [ ]:
ask_question("3.5")

In [ ]:
emissions = pd.DataFrame({zone: run.power["carbon_emission"] for zone, run in runs.items()})  # [g]
co2_per_hour = emissions.resample("1h").sum() / 1000  # [kg]
greenest = co2_per_hour.idxmin(axis=1)  # the greenest grid in each hour
switches = (greenest != greenest.shift())[1:].sum()  # migrations from one hour to the next
print(f"Greenest grid every hour: {co2_per_hour.min(axis=1).sum():,.0f} kg CO2, with {switches} migrations")

In [ ]:
give_solution("3.5", show=False)  # show=True shows the solution

In [ ]:
ask_question("3.6")

In [ ]:
half = simulate(until=TRACE_END, hosts=138, carbon="RO")
pd.DataFrame([runs["NL"].summary(), half.summary()], index=["NL, 277 hosts", "RO, 138 hosts"]).round(1)

In [ ]:
give_solution("3.6", show=False)  # show=True shows the solution

*Next (optional):* [4.live_twin.ipynb](4.live_twin.ipynb), the live digital twin.